# Notebook 03 — Geometry Import and Mesh Quality

## What you will learn
1. How to import CDB files from nTopology into MAPDL
2. Critical nTopology CDB import pitfalls (SOLID185 vs SHELL181, NROTAT)
3. What mesh quality metrics mean and why they matter
4. The Jacobian determinant — what it is and why negative = fatal
5. How to interpret aspect ratio, warping, and skewness
6. How to choose the right ANSYS element type for your physics

---

## Part 1: Geometry Import Workflow

### CDB (from nTopology)

nTopology exports meshes in the ANSYS CDB (CDWRITE) format.  The CDB is a
complete database dump containing:
- Node coordinates and IDs
- Element connectivity
- Material definitions
- Section data (for shells)
- Boundary conditions (sometimes — check before importing)

**Critical issues with nTopology CDB exports:**

| Issue | Symptom | Fix |
|-------|---------|-----|
| SOLID185 for thin shells | Shear locking — too stiff in bending | Reassign to SHELL181 |
| Node CS rotations | Wrong DOF directions after CDREAD | `NROTAT,ALL` |
| Missing real constants | Shell section thickness not imported | Define manually |
| Old nTop versions | ETYPE not set correctly | Check ETLIST before solving |

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
import matplotlib.pyplot as plt
from ams.geometry.element_selector import print_element_guide, choose_element, ELEMENT_LIBRARY

# ── Print the element reference table ─────────────────────────────────────
print_element_guide()

In [ ]:
# ── Interactive element recommendation ────────────────────────────────────
# Example: choose for an origami shell structure

rec = choose_element(
    spatial_dim       = 3,
    is_thin_shell     = True,
    large_deformation = True,
    high_accuracy     = False,
)
info = ELEMENT_LIBRARY[rec.name]

print(f'Recommended element: {rec.name}')
print(f'Best for:')
for b in info.best_for:
    print(f'  - {b}')
print(f'Avoid for:')
for a in info.avoid_for:
    print(f'  - {a}')
print(f'\nKEYOPT notes:')
for k, v in info.keyopt_notes.items():
    print(f'  KEYOPT({k}): {v}')

---

## Part 2: Mesh Quality Metrics

### Why mesh quality matters

The finite element solution converges to the true solution as the mesh is
refined — but only if the elements are **not too distorted**.

Distorted elements cause:
1. **Ill-conditioning** of the stiffness matrix (large condition number → round-off errors)
2. **Inaccurate integration** — Gauss quadrature assumes the Jacobian is smooth
3. **Solver divergence** — negative Jacobian causes the solve to immediately fail

### Jacobian determinant

For an element with nodes at positions $\mathbf{x}_i$, the Jacobian matrix $\mathbf{J}$
maps from the reference element $[{-1, 1}]^3$ to physical space:

$$J_{ij} = \frac{\partial x_i}{\partial \xi_j} = \sum_k \frac{\partial N_k}{\partial \xi_j} x_{ki}$$

The determinant $\det(\mathbf{J})$ is the local volume ratio:
- $\det(\mathbf{J}) > 0$: element is correctly oriented
- $\det(\mathbf{J}) = 0$: degenerate element (collapsed to a plane or line)
- $\det(\mathbf{J}) < 0$: **inverted element — solver WILL crash**

An inverted element means two nodes have been swapped during mesh generation.
This usually comes from STL imports with non-manifold geometry, or very
aggressive mesh refinement near sharp features.

### Aspect ratio

$$AR = \frac{\ell_{\max}}{\ell_{\min}}$$

where $\ell_{\max}$ and $\ell_{\min}$ are the longest and shortest edges of the element.

- $AR = 1$: perfect equilateral element
- $AR < 10$: acceptable for most problems
- $AR > 20$: likely to cause ill-conditioning
- $AR > 100$: degenerate — usually indicates a meshing error

High AR elements are sometimes intentional (boundary layers in CFD, thin plate
elements modeled with solid elements) — but they require careful handling.

In [ ]:
# ── Visualize element quality metrics ─────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

fig, axes = plt.subplots(2, 3, figsize=(13, 7))
axes = axes.flatten()

element_examples = [
    {
        'name': 'Perfect square\nAR=1.0, J>0',
        'nodes': np.array([[0,0],[1,0],[1,1],[0,1]]),
        'quality': 'good', 'color': 'lightgreen',
    },
    {
        'name': 'Elongated\nAR=5.0',
        'nodes': np.array([[0,0],[5,0],[5,1],[0,1]]),
        'quality': 'warn', 'color': 'lightyellow',
    },
    {
        'name': 'Skewed\nHigh skewness',
        'nodes': np.array([[0,0],[1,0],[1.5,1],[0.5,1]]),
        'quality': 'warn', 'color': 'lightyellow',
    },
    {
        'name': 'Inverted (J<0)\nFATAL — solver crash',
        'nodes': np.array([[0,0],[1,0],[0,1],[1,1]]),  # swapped top corners
        'quality': 'bad', 'color': 'salmon',
    },
    {
        'name': 'Extremely elongated\nAR=20+',
        'nodes': np.array([[0,0],[10,0],[10,0.5],[0,0.5]]),
        'quality': 'bad', 'color': 'salmon',
    },
    {
        'name': 'Good shell element\nLow warping',
        'nodes': np.array([[0,0],[1,0.05],[1.05,1.05],[0.05,1]]),
        'quality': 'good', 'color': 'lightblue',
    },
]

for ax, elem in zip(axes, element_examples):
    nodes = elem['nodes']
    # Close the polygon
    poly = np.vstack([nodes, nodes[0]])
    ax.fill(poly[:,0], poly[:,1], alpha=0.5, color=elem['color'], edgecolor='navy', linewidth=2)
    for i, (x, y) in enumerate(nodes):
        ax.plot(x, y, 'ko', markersize=6)
        ax.text(x, y, f' {i}', fontsize=9, color='black')
    ax.set_title(elem['name'], fontsize=9)
    ax.set_aspect('equal'); ax.grid(True, alpha=0.2); ax.axis('off')

plt.suptitle('Mesh Quality Examples', fontsize=11)
plt.tight_layout(); plt.show()

print('Green  = good quality  (AR < 5, J > 0.2)')
print('Yellow = marginal      (AR 5-20, J > 0)')
print('Red    = problematic   (AR > 20, or J ≤ 0)')

In [ ]:
# ── The MeshQualityChecker in action (mock data) ──────────────────────────
# In a real run, pass a live mapdl instance:
# from ams.geometry.mesh_quality import MeshQualityChecker
# checker = MeshQualityChecker(mapdl)
# report  = checker.check(raise_on_fail=True)

# Here we simulate a QualityReport for demonstration:
from ams.geometry.mesh_quality import QualityReport

mock_report = QualityReport(
    passed      = True,
    n_nodes     = 1250,
    n_elements  = 1000,
    checks      = {
        'aspect_ratio': {'status': 'PASS', 'value': 4.2, 'msg': 'max=4.20 mean=1.85 (limit 20) 0 bad elements'},
        'jacobian':     {'status': 'PASS', 'value': 0.72, 'msg': 'min=0.7200 (< 0 = inverted element — solve blocked)'},
        'warping_deg':  {'status': 'PASS', 'value': 2.1, 'msg': 'max=2.1° (shell limit 30°)'},
        'manifold':     {'status': 'PASS', 'msg': 'OK — mesh is manifold'},
        'watertight':   {'status': 'WARN', 'value': 200, 'msg': '200 free boundary faces (OK for shells; bad for solids)'},
    }
)
mock_report.print_summary()

---

## Summary

| Metric | Good | Warning | Fatal |
|--------|------|---------|-------|
| Aspect ratio | < 5 | 5–20 | > 20 |
| Jacobian | > 0.5 | 0–0.5 | ≤ 0 |
| Warping (shells) | < 10° | 10–30° | > 30° |
| Manifold | Yes | — | No (geometry error) |

**Always run mesh quality check before solving.** The `MeshQualityChecker` is
gate-checked before `run_solution()` — it will raise RuntimeError if critical
thresholds are violated.

**Next:** `04_boundary_conditions.ipynb`